# 300 m Walkability: Isochrone Computation (DuckDB version)
- For each residential address: determine whether it lies within **300 m walking distance** of a publicly accessible park or garden 
- Park/garden = define as any size public greenery in the polygon of given city. 
- Public restrain is given by 1.OSM labels, 2.walking path x park border crossing

* for gh algo: run config/run_gh.sh {city}

In [ ]:
CITY    = "bologna"          

#=======================================

import warnings
warnings.filterwarnings("ignore")

import time
from pathlib import Path

import duckdb
import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd
import requests
from shapely.geometry import MultiPoint, Point, shape
from shapely.ops import nearest_points
from shapely.strtree import STRtree
from shapely.validation import make_valid
import os
import sys
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm
from polylabel import polylabel
import pyproj
from shapely.ops import transform as shp_transform
from config.config import (EUCLID_CUTOFF, connect_db, gdf_to_duckdb, load_table, gpkg_to_duckdb,get_city_cfg, get_output_paths,
        DB_PATH, GH_PORT, CRS_WGS, CRS_METRIC, WALKING_DISTANCE, ROOT_PATH,
        OSM_GREEN_TAGS, PRIVATE_ACCESS,

)

sys.path.insert(0, str(ROOT_PATH))
os.chdir(ROOT_PATH)

con = connect_db()

cfg     = get_city_cfg(CITY)
src     = cfg["data_sources"]
paths   = get_output_paths(CITY)

GH_URL  = f"http://localhost:{GH_PORT}/route"
ISO_URL = f"http://localhost:{GH_PORT}/isochrone"

print(f"City: {CITY}")
print(f"GH URL: {GH_URL}")
print(f"DB path: {DB_PATH}")

OSM_SNAP_TOL = 2.0   # OSM node must be within 2 m of park boundary

ModuleNotFoundError: No module named 'duckdb'

In [3]:
con.close()

### Data: DuckDB

In [ ]:
parks = load_table(con, "preprocessing_2", f"{CITY}_parks_greenery")
parks["geometry"] = parks["geometry"].apply(make_valid)
parks = parks[parks["area_ha"] >= 1.0]
print(f"Parks  : {len(parks):,}  CRS={parks.crs}")

result = con.execute(f"""
    SELECT
        source,
        COUNT(*)        AS n_parks,
        SUM(area_ha)    AS total_ha
    FROM preprocessing_2.{CITY}_parks_greenery
    WHERE area_ha >= 1.0
    GROUP BY source
""").fetchall()

for row in result:
    print(f"{row[0]:<40} {row[1]:>6,} parks   {row[2]:>8.0f} ha")
   
boundary = load_table(con, "raw", f"{CITY}_boundary")
boundary_wgs84 = boundary.to_crs(CRS_WGS).geometry.union_all()
print(f"Boundary area  : {boundary_wgs84.area:.6f} deg²")
print(f"Boundary bounds: {boundary_wgs84.bounds}")

# "buildings_sn" key  => raw.milano_buildings_sn
civici = load_table(con, "raw", f"{CITY}_buildings_sn")

# Use pre-computed building_id if it exists, else derive it - this is important as it differe between data for different cities
if "building_id" in civici.columns:
    civici["building_id"] = civici["building_id"].astype(str)
else:
    civici["building_id"] = (
        civici["CODICE_VIA"].astype(str).str.strip() + "_"
        + civici[cfg["building_id_col"]].astype(str).str.strip() + "_"
        + civici.get("LETTERA", pd.Series("", index=civici.index)).fillna("").astype(str)
    )

civici = civici.reset_index(drop=True)
assert civici["building_id"].nunique() == len(civici), \
    f"building_id not unique! {civici['building_id'].nunique()} unique / {len(civici)} rows"

Parks  : 604  CRS=EPSG:32632
verona_parks_osm                            604 parks       4218 ha
Boundary area  : 0.022867 deg²
Boundary bounds: (10.878436411535013, 45.350409962699416, 11.12388124678979, 45.54153478471377)


: 

### Walking distance | detect park entrances


In [3]:
print("Downloading walk network: ")
G = ox.graph_from_polygon(boundary_wgs84, network_type="walk")
G = ox.project_graph(G, to_crs=CRS_METRIC)
_, edges = ox.graph_to_gdfs(G)
edges = edges.reset_index(drop=True)

print(f"Pedestrian edges from OSM: {len(edges):,}  CRS={edges.crs}")

print("Get OSM entrance: ")
osm_entrances_raw = ox.features_from_polygon(
    boundary_wgs84,
    tags={"entrance": ["yes", "main", "secondary", "service",
                       "staircase", "emergency", "exit"]},
).to_crs(CRS_METRIC)

osm_entrance_pts = osm_entrances_raw[
    osm_entrances_raw.geometry.geom_type == "Point"
].copy()
print(f"Number of OSM entrance: {len(osm_entrance_pts):,}")


def extract_points(geom) -> list:
    """Extract all Point objects from any Shapely geometry type."""
    if geom is None or geom.is_empty:
        return []
    t = geom.geom_type
    if t == "Point":
        return [geom]
    if t == "MultiPoint":
        return list(geom.geoms)
    if t == "GeometryCollection":
        pts = []
        for g in geom.geoms:
            pts.extend(extract_points(g))
        return pts
    return []

def get_boundary_entrance(geom, edge_tree, edges_geom, search_radius=50):
    """
    Sample boundary at regular intervals, find closest edge point.
    More honest than centroid — places the entrance ON the boundary
    near actual infrastructure.
    """
    boundary = geom.boundary
    # Sample every 10m along boundary
    length = boundary.length
    n_samples = max(8, int(length / 10))
    samples = [boundary.interpolate(i / n_samples, normalized=True) 
               for i in range(n_samples)]
    
    best_pt, best_dist = None, float("inf")
    for pt in samples:
        hits = edge_tree.query(pt.buffer(search_radius), predicate="intersects")
        for idx in hits:
            edge_pt, _ = nearest_points(edges_geom[idx], pt)
            d = pt.distance(edge_pt)
            if d < best_dist:
                best_dist = d
                best_pt = pt  # snap to boundary, not to edge
    return best_pt

edges_geom = edges.geometry.values
edge_mids  = np.array([
    [g.interpolate(0.5, normalized=True).x,
     g.interpolate(0.5, normalized=True).y]
    for g in edges_geom
])

osm_entrance_tree = STRtree(osm_entrance_pts.geometry.values)
edge_tree = STRtree(edges_geom)

rows = []
step_counts = {1: 0, 2: 0, 3: 0, "3_extended": 0}

for _, park in tqdm(parks.iterrows(), total=len(parks), desc="Park entrances"):
    geom      = park.geometry
    park_bdry = geom.boundary      # per-park LineString 
    park_id   = park["park_id"]
    pts       = []

    # Tier 1: explicit OSM entrance=* node within OSM_SNAP_TOL of boundary
    osm_hits  = osm_entrance_tree.query(geom.buffer(OSM_SNAP_TOL),
                                        predicate="intersects")
    tier1_pts = [
        osm_entrance_pts.geometry.iloc[i]
        for i in osm_hits
        if park_bdry.distance(osm_entrance_pts.geometry.iloc[i]) <= OSM_SNAP_TOL
    ]
    if tier1_pts:
        pts = tier1_pts
        step_counts[1] += 1

    # Tier 2: boundary ∩ external edges + near-miss projection
    if not pts:
        tier2_raw = []
        for idx in edge_tree.query(park_bdry, predicate="intersects"):
            if not geom.contains(edges_geom[idx]):
                inter = park_bdry.intersection(edges_geom[idx])
                tier2_raw.extend(extract_points(inter))

        if tier2_raw:
            pts = tier2_raw
            step_counts[2] += 1

    if not pts:
        if park["source"] == f"{CITY}_parks_osm" or (pd.notna(park.get("name")) and park.get("name")):
            snap = get_boundary_entrance(geom, edge_tree, edges_geom)
            #snap = get_interior_point(geom)

            if snap:
                pts.append(snap)
                if park["source"] == f"{CITY}_parks_osm":
                    step_counts[3] += 1
                else:
                    step_counts["3_extended"] += 1

    transformer = pyproj.Transformer.from_crs(CRS_METRIC, CRS_WGS, always_xy=True)

    for pt in pts:
        lon, lat = transformer.transform(pt.x, pt.y)
        rows.append({
            "park_id":      park_id,
            "entrance_lat": lat,
            "entrance_lon": lon,
            "geometry":     pt,
        })


entrances_df = gpd.GeoDataFrame(rows, geometry="geometry", crs=CRS_METRIC)

missing = set(parks["park_id"]) - set(entrances_df["park_id"])
print(f"\n{len(entrances_df):,} entrance points across {entrances_df['park_id'].nunique()} parks")
print(f"OSM: {step_counts[1]} | boundary∩street: {step_counts[2]} | "
      f"centroid: {step_counts[3]} | extended: {step_counts['3_extended']}")
print(f"Parks with no entrance: {len(missing)}")

gdf_to_duckdb(con, entrances_df, f"preprocessing_2.{CITY}_entrances")

#check parks without entrance
#if missing:
    #print("  Missing IDs:", sorted(missing))

civici_in_park = con.execute(f"""
    SELECT c.building_id, p.park_id
    FROM raw.{CITY}_buildings_sn c
    JOIN preprocessing_2.{CITY}_parks_greenery p
      ON ST_Within(c.geometry, p.geometry)
""").df()

civici_in_park_ids    = set(civici_in_park["building_id"])
civici_in_park_lookup = civici_in_park.set_index("building_id")["park_id"].to_dict()
print(f"Civici inside a park polygon: {len(civici_in_park_ids):,}")


Pedestrian edges from OSM: 55,838  CRS=EPSG:32632
Get OSM entrance: 
Number of OSM entrance: 370


Park entrances: 100%|██████████| 604/604 [00:15<00:00, 40.22it/s]



5,389 entrance points across 543 parks
OSM: 6 | boundary∩street: 313 | centroid: 224 | extended: 0
Parks with no entrance: 61
preprocessing_2.verona_entrances: 5389 rows written
Civici inside a park polygon: 250


### GraphHopper (isochrones)

In [4]:
#walking isochrone from GraphHopper
#Returns = polygon feature dict

def get_isochrone(lat: float, lon: float, distance_m: int = 300):
    params = {
        "point":          f"{lat},{lon}",
        "profile":        "foot",
        "distance_limit": distance_m,
        "type":           "json",
    }
    try:
        r = requests.get(ISO_URL, params=params, timeout=10)
        r.raise_for_status()
        return r.json()["polygons"][0]
    except Exception:
        return None
    
iso_rows = []
for _, row in tqdm(entrances_df.iterrows(), total=len(entrances_df), desc="Isochrones"):
    result = get_isochrone(row["entrance_lat"], row["entrance_lon"])
    if result:
        iso_rows.append({
            "park_id":      row["park_id"],
            "entrance_lat": row["entrance_lat"],
            "entrance_lon": row["entrance_lon"],
            "geometry":     shape(result["geometry"]),
        })

# Isochrones from GH are WGS84 = reproject to metric for spatial join!
isochrones = gpd.GeoDataFrame(
    iso_rows,
    geometry="geometry",
    crs=CRS_WGS,
).to_crs(CRS_METRIC)

print(f"{len(isochrones):,} isochrones computed"
      f"({len(entrances_df) - len(isochrones):,} failed)")

#save intermediate results to
gdf_to_duckdb(con, isochrones, f"preprocessing_2.{CITY}_isochrones")


Isochrones: 100%|██████████| 5389/5389 [06:39<00:00, 13.49it/s] 


5,362 isochrones computed(27 failed)
preprocessing_2.verona_isochrones: 5362 rows written


### Spatial join: civici within isochrones | Point-to-point routing (GraphHopper)

In [5]:
civici_gdf = civici[["building_id", "geometry"]].copy().reset_index(drop=True)

joined = gpd.sjoin(
    civici_gdf,
    isochrones[["park_id", "entrance_lat", "entrance_lon", "geometry"]],
    how="left",
    predicate="within",
)

candidates = joined[joined["park_id"].notna()].copy().reset_index(drop=True)

print(f"Raw join rows : {len(joined):,}")
print(f"Matched (in iso) : {len(candidates):,}")
print(f"Unmatched : {joined['park_id'].isna().sum():,}")
print(f"Unique civici matched: {candidates['building_id'].nunique():,} "
      f"({candidates['building_id'].nunique()/len(civici)*100:.1f}%)")

cands_wgs = gpd.GeoDataFrame(
    candidates[["building_id"]],
    geometry=candidates.geometry,
    crs=CRS_METRIC,
).to_crs(CRS_WGS) # WGS84 coords of each candidate civico (for GH routing)

candidates = candidates.copy()
candidates["orig_lat"] = cands_wgs.geometry.y
candidates["orig_lon"] = cands_wgs.geometry.x

#Euclidean pre-filter - 420 m - so it is lighter to process the routing later
entrance_pts_metric = gpd.GeoDataFrame(
    candidates[["entrance_lat", "entrance_lon"]],
    geometry=gpd.points_from_xy(candidates["entrance_lon"], candidates["entrance_lat"]),
    crs=CRS_WGS,
).to_crs(CRS_METRIC)

candidates["euclid_m"] = candidates.geometry.distance(entrance_pts_metric.geometry)

before = len(candidates)
candidates = (
    candidates[candidates["euclid_m"] <= 320]
    .copy()
    .reset_index(drop=True)
)
after = len(candidates)
print(f"Euclidean pre-filter: {before:,} = {after:,} "
      f"(dropped {before-after:,}, kept {after/before*100:.1f}%)")

# Per-civico cap: keep only the N closest entrances = lighter processing 
MAX_ENTRANCES_PER_CIVICO = 1
candidates = (
    candidates
    .sort_values("euclid_m")
    .groupby("building_id", sort=False)
    .head(MAX_ENTRANCES_PER_CIVICO)
    .reset_index(drop=True)
)

n_pairs = len(candidates)
n_cands = candidates["building_id"].nunique()
print(f"{n_cands:,} candidate civici ({n_cands/len(civici)*100:.1f}% of total)")
print(f"{n_pairs:,} (civico, entrance) pairs to route")
print(f"avg {n_pairs/max(n_cands,1):.1f} entrance candidates per civico")

def graphhopper_dist(orig_lat, orig_lon, dest_lat, dest_lon):
    params = {
        "point":        [f"{orig_lat},{orig_lon}", f"{dest_lat},{dest_lon}"],
        "profile":      "foot",
        "calc_points":  "false",
        "instructions": "false",
    }
    try:
        r = requests.get(GH_URL, params=params, timeout=5)
        r.raise_for_status()
        return r.json()["paths"][0]["distance"]
    except Exception:
        return None

rows = []
for _, row in tqdm(candidates.iterrows(), total=n_pairs, desc="Routing"):
    dist = graphhopper_dist(
        row["orig_lat"], row["orig_lon"],
        row["entrance_lat"], row["entrance_lon"],
    )
    if dist is not None:
        rows.append((row["building_id"], row["park_id"],
                     row["entrance_lat"], row["entrance_lon"], dist))

print(f"{len(rows):,} pairs routed  ({n_pairs - len(rows):,} GH failures)")

routing_df = pd.DataFrame(rows, columns=[
    "building_id", "park_id", "entrance_lat", "entrance_lon", "walk_dist_m"
])
routing_df["city"] = CITY

con.register("routing_tmp_2", routing_df)
con.execute(f"""
    CREATE OR REPLACE TABLE rule_300m_2.{CITY}_routing AS
    SELECT * FROM routing_tmp_2
""")
con.unregister("routing_tmp_2")

best = con.execute(f"SELECT * FROM rule_300m_2.{CITY}_routing").df()
print(f"{len(best):,} unique civici with a best route")
print(best["walk_dist_m"].describe().round(1).to_string())

Raw join rows : 91,559
Matched (in iso) : 49,735
Unmatched : 41,824
Unique civici matched: 11,101 (21.0%)
Euclidean pre-filter: 49,735 = 49,059 (dropped 676, kept 98.6%)
11,041 candidate civici (20.9% of total)
11,041 (civico, entrance) pairs to route
avg 1.0 entrance candidates per civico


Routing: 100%|██████████| 11041/11041 [02:19<00:00, 79.27it/s] 


11,041 pairs routed  (0 GH failures)
11,041 unique civici with a best route
count    11041.0
mean       201.5
std        201.3
min          0.0
25%        125.2
50%        200.2
75%        260.9
max      10018.3


### Output

In [6]:
out_df = con.execute(f"""
    SELECT
        c.*,
        b.park_id       AS nearest_park_id,
        b.entrance_lat,
        b.entrance_lon,
        b.walk_dist_m,
        b.walk_dist_m IS NOT NULL              AS is_candidate,
        COALESCE(b.walk_dist_m <= {WALKING_DISTANCE}, FALSE) AS within_300m
    FROM raw.{CITY}_buildings_sn c
    LEFT JOIN rule_300m_2.{CITY}_routing b USING (building_id)
""").df()

out_df["geometry"] = out_df["geometry"].apply(bytes)
out = gpd.GeoDataFrame(
    out_df,
    geometry=gpd.array.from_wkb(out_df["geometry"]),
    crs=CRS_METRIC,
)

# Apply civici_in_park overrides
in_park_mask = out["building_id"].isin(civici_in_park_ids)
out.loc[in_park_mask, "within_300m"] = True
out.loc[in_park_mask, "walk_dist_m"] = out.loc[in_park_mask, "walk_dist_m"].fillna(0.0)
out.loc[in_park_mask, "nearest_park_id"] = (
    out.loc[in_park_mask, "building_id"]
    .map(civici_in_park_lookup)
    .combine_first(out.loc[in_park_mask, "nearest_park_id"])
)

# Rename
rename = {
    cfg["building_id_col"]: "street_number",
    "indirizzo_completo":   "address",
}
if cfg["neighbourhood_col"] and cfg["neighbourhood_col"] in out.columns:
    rename[cfg["neighbourhood_col"]] = "neighbourhood"
out = out.rename(columns=rename)

# DuckDB
gdf_to_duckdb(con, out, f"rule_300m_2.{CITY}_results")

# WGS84
out_wgs = out.to_crs(CRS_WGS)
out_wgs["longitude"] = out_wgs.geometry.x
out_wgs["latitude"]  = out_wgs.geometry.y

final_cols = [
    "street_number", "address", "longitude", "latitude",
    "neighbourhood", "nearest_park_id",
    "park_area_ha", "entrance_lat", "entrance_lon",
    "walk_dist_m", "within_300m", "geometry",
]
out_wgs = out_wgs[[c for c in final_cols if c in out_wgs.columns]]

# GeoPackage
paths = get_output_paths(CITY)
paths["gpkg"] = paths["gpkg"].replace(".gpkg", "_2.gpkg")

out.to_file(paths["gpkg"], driver="GPKG", layer="buildings")
parks[parks["park_id"].isin(out["nearest_park_id"].dropna())].to_file(
    paths["gpkg"], driver="GPKG", layer="parks"
)

# Exports
parquet_path = paths["parquet"]  # already absolute from get_output_paths()
out_wgs.to_parquet(parquet_path, index=False)

con.execute(f"""
    CREATE OR REPLACE VIEW results_300m_2.{CITY}_geoparquet AS
    SELECT * FROM read_parquet('{parquet_path}')
""")

print(f"{CITY} view registered")

con.close()

rule_300m_2.verona_results: 52925 rows written
verona view registered
